# CS 691 · Can a language model answer from the current record?

Ava is looking for the desk that will hand her a camera. Her question sounds easy until you discover that one record describes where a camera is stored, another describes where it is collected, and a third applies to a different day.

**Goal.** Retrieve the relevant records, give them to a pretrained language model, then decide which answer the evidence supports. By the end, you should be able to explain how a system can decline a confident answer and how it can attach a checkable citation to a correct one.

Run the cells in order. The records, prompt builder and verifier use ordinary Python. To generate a new answer with Google's FLAN-T5-small, turn on the switch near the end. Only that section needs an internet connection for the initial package and model download. A CPU is sufficient. No API key or model account is needed.

## Read the desk records

Each row describes one camera on one day. This fictional desk assigns each camera **one pickup desk per day**. A storage location answers a different question. Two equally current pickup records naming different desks leave a conflict for the clerk to resolve.

**Predict first.** What can we tell Ava about C1 on Monday? What could go wrong if we answered for C2 using the word “desk” alone?

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Record:
    source: str
    item: str
    day: str
    relation: str
    value: str

records = [
    Record("D8", "C1", "Monday", "pickup", "West Desk"),
    Record("S2", "C1", "Monday", "storage", "North Desk"),
    Record("D10", "C1", "Tuesday", "pickup", "East Desk"),
    Record("S4", "C2", "Monday", "storage", "East Desk"),
    Record("D11", "C3", "Monday", "pickup", "North Desk"),
    Record("D12", "C3", "Monday", "pickup", "West Desk"),
]

questions = [("C1", "Monday"), ("C2", "Monday"), ("C3", "Monday")]
for record in records:
    print(record.source, record.day, record.item, record.relation, record.value)

## Retrieve the context and prepare the prompt

The desk form supplies the item ID and the day as separate fields. Retrieval selects rows with those two values. We keep both pickup and storage rows so we can see whether the model answers the question that was asked. A **prompt** is the text the model receives, including the question, instructions and records. Look at the exact prompt before running any model.

In [ ]:
def retrieve(item, day, ledger):
    return [row for row in ledger if row.item == item and row.day == day]

def make_prompt(item, day, context):
    lines = ["Read the records. If the answer is missing or the records disagree, answer unknown."]
    for row in context:
        lines.append(f"{row.source} says the {row.relation} desk for {row.item} "
                     f"on {row.day} is {row.value}.")
    if not context:
        lines.append("There are no records for that camera and day.")
    lines.append(f"Question: What is the pickup desk for {item} on {day}?")
    lines.append("Answer:")
    return "\n".join(lines)

print(make_prompt("C1", "Monday", retrieve("C1", "Monday", records)))

## Decide what the evidence permits

For this one question, the entire answer can be a single desk name. The parser reads that whole short answer, allowing differences in case or a final period. It recognizes the desk names in this exercise and the word “unknown.” Extra words could introduce a second claim, so a human would need to review a longer reply.

The verifier independently checks the requested item, day and relation. All matching pickup records must agree. When the model's desk matches, the verifier attaches the IDs of the records that establish it. When evidence is absent or conflicts, the verifier withholds the desk name even if the model guessed one. A model decline stays a decline.

In [ ]:
desk_names = {name.casefold(): name for name in ("West Desk", "East Desk", "North Desk")}

def parse_draft(raw):
    text = " ".join(raw.strip().removesuffix(".").split()).casefold()
    if text == "unknown":
        return "unknown"
    return desk_names.get(text)

def verify(item, day, raw, context):
    pickup = [row for row in context if row.item == item and row.day == day
              and row.relation == "pickup"]
    values = {row.value for row in pickup}
    if not pickup:
        return "abstain", f"No pickup record for {item} on {day}."
    if len(values) > 1:
        evidence = ", ".join(f"{r.source} says {r.value}" for r in pickup)
        return "abstain", "Conflicting pickup records: " + evidence

    claim = parse_draft(raw)
    if claim == "unknown":
        return "abstain", "The model declined to name a desk."
    if claim is None:
        return "abstain", "The draft is outside the single-desk answer format."
    if claim not in values:
        return "abstain", "The proposed desk disagrees with the pickup record."
    sources = ", ".join(row.source for row in pickup)
    return "publish", f"Collect {item} at {claim} on {day} [{sources}]."

## Test the checker with supplied drafts

These are **hand-written test inputs**. None is a recorded model prediction. Predict where the checker will publish or abstain, then compare your reasoning with its printed explanation. The last case deliberately hands the verifier records for the wrong day to test that it checks the date itself.

In [ ]:
tests = [
    ("C1", "Monday", "West Desk.", "publish"),
    ("C1", "Monday", "North Desk", "abstain"),          # storage
    ("C1", "Monday", "East Desk", "abstain"),           # another day
    ("C1", "Monday", "West Desk before noon", "abstain"),
    ("C1", "Monday", "unknown", "abstain"),
    ("C2", "Monday", "East Desk", "abstain"),           # missing pickup
    ("C3", "Monday", "North Desk", "abstain"),          # conflicting records
    ("C1", "Tuesday", "East Desk", "publish"),
]
for item, day, supplied_draft, expected in tests:
    result, reason = verify(item, day, supplied_draft, retrieve(item, day, records))
    assert result == expected, (item, day, supplied_draft, reason)
    print(f"Supplied draft {supplied_draft!r} → {result}: {reason}")

wrong_day = retrieve("C1", "Tuesday", records)
result, reason = verify("C1", "Monday", "East Desk", wrong_day)
assert result == "abstain" and "No pickup record" in reason
print(f"Wrong-day retrieval → {result}: {reason}")

## Generate a live draft with FLAN-T5-small

This **encoder–decoder** model reads the prompt into a numerical representation and uses it to generate an answer one piece at a time. Generation uses the pretrained weights. The checkout records enter through the prompt, so changing a record changes what the model sees without retraining it.

In Colab, set `RUN_PRETRAINED = True` below and run this cell and the next two. The first use downloads the tokenizer and model weights from Google’s public Hugging Face repository. The tokenizer converts between text and the model’s numbers. The model revision is fixed so everyone requests the same weights. The generated text is printed exactly as decoded before the independent verifier sees it.

[Google’s model card and CPU example](https://huggingface.co/google/flan-t5-small) · [Transformers documentation for FLAN-T5](https://huggingface.co/docs/transformers/model_doc/flan-t5)

In [ ]:
RUN_PRETRAINED = False  # Change to True for the model download and live generation.

if RUN_PRETRAINED:
    from importlib.util import find_spec
    packages = "transformers==4.57.6 sentencepiece>=0.2,<0.3"
    if find_spec("torch") is None:
        packages += " torch>=2.4"
    # %pip installs into the Python environment running this notebook.
    get_ipython().run_line_magic("pip", 'install -q ' + " ".join(
        f'"{package}"' for package in packages.split()
    ))
else:
    print("Offline exercises are ready. Live generation is switched off.")

In [ ]:
if RUN_PRETRAINED:
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    model_id = "google/flan-t5-small"
    revision = "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab"
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id, revision=revision)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    print("Loaded", model_id, "on", device)

In [ ]:
if RUN_PRETRAINED:
    for item, day in questions:
        context = retrieve(item, day, records)
        prompt = make_prompt(item, day, context)
        tokens = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.inference_mode():
            generated = model.generate(**tokens, do_sample=False, max_new_tokens=24)
        draft = tokenizer.decode(generated[0], skip_special_tokens=True)
        result, reason = verify(item, day, draft, context)
        print(f"\nQuestion: Where do I collect {item} on {day}?")
        print("Records sent:", ", ".join(row.source for row in context))
        print("Live draft:", repr(draft))
        print(f"Checked answer: {result} — {reason}")

## Check your understanding

1. The model answers “East Desk” for C2. Why is this a plausible response, and why should the verifier abstain?
2. For C3, what new evidence would allow the desk to answer? Would a third model guess settle the disagreement?
3. Replace C1’s Monday pickup record with a new current record that says East Desk and has a fresh ID. Repeat retrieval and verification. What must change for the verifier to publish? If you run the model, record separately whether its draft changes.

<details><summary>Reveal worked answers</summary>

1. S4 names East Desk as C2’s storage location. No row names a C2 pickup location on Monday. The system can ask the clerk for that specific record.
2. D11 and D12 are equally current, both concern C3 on Monday, and give incompatible pickup locations. An authoritative correction identifying the valid assignment and superseding the other record would resolve the conflict. Another model guess supplies no new desk record.
3. The new Monday record must replace the old one. Keeping both with equal authority produces a conflict. A single-desk draft of “East Desk” will now pass and cite the replacement ID. A draft of “West Desk” will be withheld. The real model’s reply is an observation to make in the live cell.

</details>